<a href="https://colab.research.google.com/github/separdoc-byte/-ndice-del-sancocho/blob/main/El_%C3%ADndice_sancocho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# SESIÓN 1. LA RECETA COMO FUNCIÓN

***

## Proyecto: El índice sancocho

**Asignatura:** Cálculo Diferencial  
**Tema:** Funciones, datos y procesamiento de precios.

### Objetivo

Representar la receta de un sancocho para seis personas mediante estructuras de datos y funciones de Python, con el propósito de calcular su costo a partir de los precios por kilogramo registrados en SIPSA y comparar los resultados con nuestras predicciones iniciales.

### Metodología

Organizaremos la receta, prepararemos los datos, identificaremos los precios de los ingredientes, construiremos un pipeline de procesamiento y analizaremos el costo estimado de la preparación.

***************************

# 1. Importación de bibliotecas
***
Importaremos las bibliotecas necesarias para poder poner en funcionamiento varios datos de nuestro código, los cuales dependen de estas para ejecutarse.

In [ ]:
import pandas as pd
import unicodedata
import zipfile
from pathlib import Path
from IPython.display import display

## 2. Carga y lectura del archivo de datos SIPSA
***
En esta sección cargamos el archivo de datos del Sistema de Información de Precios y Abastecimiento del Sector Agropecuario (SIPSA), publicado por el DANE. Este archivo contiene información sobre los precios de los productos agropecuarios y será la fuente de datos para calcular el costo estimado de los ingredientes del sancocho en Manizales durante diciembre de 2024.

In [ ]:
ruta_zip = "/content/BaseDatos-SIPSA_P-Mensual-2024.zip"

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:
    print("Archivos dentro del ZIP:")
    for nombre in archivo_zip.namelist():
        print("-", nombre)

    # El archivo CSV del ZIP se llama "mensual 24.csv"
    with archivo_zip.open("mensual 24.csv") as archivo_csv:
        datos = pd.read_csv(
            archivo_csv,
            sep=";",
            encoding="utf-8-sig",
            dtype=str
        )

print("Dimensiones de los datos:", datos.shape)
print("Columnas:", datos.columns.tolist())
display(datos.head())

Archivos dentro del ZIP:
- mensual 24.csv
- Mensual_24.dta
- mensual_24.sav
Dimensiones de los datos: (54042, 6)
Columnas: ['Fecha', 'Grupo', 'Producto', 'CODIGO_CPC_AC', 'Mercado', 'Precio promedio por kilogramo*']


,Fecha,Grupo,Producto,CODIGO_CPC_AC,Mercado,Precio promedio por kilogramo*
0,ene-24,VERDURAS Y HORTALIZAS,Acelga,0121903,"Armenia, Mercar",1.387
1,ene-24,VERDURAS Y HORTALIZAS,Acelga,0121903,"Bogotá, D.C., Corabastos",607
2,ene-24,VERDURAS Y HORTALIZAS,Acelga,0121903,"Bucaramanga, Centroabastos",2.000
3,ene-24,VERDURAS Y HORTALIZAS,Acelga,0121903,Chiquinquirá (Boyacá),1.896
4,ene-24,VERDURAS Y HORTALIZAS,Acelga,0121903,"Duitama (Boyacá), Mercaplaza",2.000


## 3. Limpieza y preparación de los datos
***

En esta sección se realiza la preparación de la base de datos SIPSA para facilitar su análisis. Se revisan los nombres de las columnas, los tipos de datos y los posibles valores faltantes. Además, se convierten los precios a formato numérico para permitir realizar operaciones matemáticas con ellos.

Este proceso es importante porque permite organizar la información y evitar errores en los cálculos posteriores del costo de los ingredientes del sancocho. De esta manera, los datos quedan preparados para filtrar los productos, seleccionar los precios correspondientes a Manizales y calcular el costo total de la receta.

In [ ]:
# Normalizar el texto para facilitar comparaciones
def normalizar(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

# Convertir precios como "13.500" a 13500
def convertir_precio(valor):
    if pd.isna(valor):
        return float("nan")
    texto = str(valor).strip()
    texto = texto.replace(".", "").replace(",", ".")
    return pd.to_numeric(texto, errors="coerce")

datos["precio_kg"] = datos["Precio promedio por kilogramo*"].apply(convertir_precio)
datos["fecha_normalizada"] = datos["Fecha"].str.strip().str.lower()
datos["mercado_normalizado"] = datos["Mercado"].apply(normalizar)
datos["producto_normalizado"] = datos["Producto"].apply(normalizar)

print("Ejemplo de precios preparados:")
display(datos[["Fecha", "Producto", "Mercado", "precio_kg"]].head())

Ejemplo de precios preparados:


,Fecha,Producto,Mercado,precio_kg
0,ene-24,Acelga,"Armenia, Mercar",1387
1,ene-24,Acelga,"Bogotá, D.C., Corabastos",607
2,ene-24,Acelga,"Bucaramanga, Centroabastos",2000
3,ene-24,Acelga,Chiquinquirá (Boyacá),1896
4,ene-24,Acelga,"Duitama (Boyacá), Mercaplaza",2000



## 4. Construcción de la receta

***

Representamos la receta del sancocho para seis personas mediante una tabla. Cada ingrediente tiene una cantidad expresada en kilogramos, lo que permitirá relacionarlo posteriormente con su precio por kilogramo.


In [ ]:
# Receta oficial del sancocho para 6 personas
receta = pd.DataFrame([
    {"ingrediente": "espinazo", "cantidad_kg": 1.5},
    {"ingrediente": "yuca", "cantidad_kg": 1.0},
    {"ingrediente": "papa", "cantidad_kg": 1.0},
    {"ingrediente": "mazorca", "cantidad_kg": 0.9},
    {"ingrediente": "platano", "cantidad_kg": 1.4}
])

print("RECETA DE SANCOCHO PARA 6 PERSONAS")
display(receta)

print("Peso total de los ingredientes:",
      receta["cantidad_kg"].sum(), "kg")


RECETA DE SANCOCHO PARA 6 PERSONAS


,ingrediente,cantidad_kg
0,espinazo,1.5
1,yuca,1.0
2,papa,1.0
3,mazorca,0.9
4,platano,1.4


Peso total de los ingredientes: 5.800000000000001 kg



## 5. Definición de las funciones de costo

***
Creamos funciones reutilizables para calcular el costo de cada ingrediente y el costo total de la receta. El costo individual se obtiene multiplicando la cantidad requerida por el precio por kilogramo, mientras que el costo total corresponde a la suma de los costos individuales.


In [ ]:

def calcular_costo_ingrediente(cantidad_kg, precio_kg):
    return cantidad_kg * precio_kg


def calcular_costo_total(receta_con_precios):
    return receta_con_precios["costo"].sum()


## 6. Prueba de las funciones con datos de ejemplo

***

Antes de utilizar los precios de SIPSA, comprobamos el funcionamiento de las funciones con valores ficticios. Esta prueba permite verificar que el cálculo es correcto. Los precios de esta sección son únicamente ilustrativos y no representan valores oficiales del DANE.


In [ ]:

# Crear una copia de la receta para hacer la prueba
ejemplo = receta.copy()

# Precios ficticios por kilogramo
ejemplo["precio_kg"] = [15000, 3000, 2500, 4000, 2800]

# Calcular el costo de cada ingrediente
ejemplo["costo"] = ejemplo.apply(
    lambda fila: calcular_costo_ingrediente(
        fila["cantidad_kg"],
        fila["precio_kg"]
    ),
    axis=1
)

display(ejemplo)

costo_ejemplo = calcular_costo_total(ejemplo)

print(f"Costo total del ejemplo: ${costo_ejemplo:,.0f} COP")


,ingrediente,cantidad_kg,precio_kg,costo
0,espinazo,1.5,15000,22500.0
1,yuca,1.0,3000,3000.0
2,papa,1.0,2500,2500.0
3,mazorca,0.9,4000,3600.0
4,platano,1.4,2800,3920.0


Costo total del ejemplo: $35,520 COP


#7. Filtrado de Manizales y diciembre de 2024
***
El archivo contiene registros de distintos mercados y meses. Filtramos el mercado Manizales, Centro Galerías y el mes dic-24.

In [ ]:
datos_filtrados = datos[
    (datos["fecha_normalizada"] == "dic-24") &
    (datos["mercado_normalizado"] == normalizar("Manizales, Centro Galerías"))
].copy()

print("Cantidad de registros de Manizales, diciembre de 2024:", len(datos_filtrados))
display(datos_filtrados[["Fecha", "Producto", "Mercado", "precio_kg"]].head())

Cantidad de registros de Manizales, diciembre de 2024: 111


,Fecha,Producto,Mercado,precio_kg
49559,dic-24,Ahuyama,"Manizales, Centro Galerías",1504
49600,dic-24,Ajo,"Manizales, Centro Galerías",10708
49626,dic-24,Ajo importado,"Manizales, Centro Galerías",11115
49654,dic-24,Apio,"Manizales, Centro Galerías",1897
49686,dic-24,Arveja verde en vaina,"Manizales, Centro Galerías",5521



#8. Identificación de los precios de los ingredientes
***
Relacionamos cada ingrediente con el nombre exacto registrado en SIPSA y buscamos su precio promedio por kilogramo para el mercado y mes seleccionados.


In [ ]:
# Verificar los productos que se van a utilizar
productos_necesarios = receta["producto_sipsa"].tolist()

for producto in productos_necesarios:
    coincidencias = datos_filtrados[
        datos_filtrados["producto_normalizado"] == normalizar(producto)
    ]
    if coincidencias.empty:
        print("NO ENCONTRADO:", producto)
    else:
        print(producto, "->", coincidencias["precio_kg"].tolist())

Carne de cerdo, espinazo -> [13500]
Yuca ICA -> [1377]
Papa capira -> [2133]
Chócolo mazorca -> [1273]
Plátano dominico hartón verde -> [2663]


#9. Construcción del pipeline
***
El pipeline une la receta con los precios encontrados, calcula el costo por ingrediente y devuelve una tabla final. Si falta algún precio, se informa para evitar presentar un total incompleto.

In [ ]:

def construir_pipeline(datos_filtrados, receta):
    # Promedio de los registros disponibles por nombre de producto.
    # En este archivo mensual normalmente hay un registro por producto y mercado.
    precios = (
        datos_filtrados
        .groupby("producto_normalizado", as_index=False)["precio_kg"]
        .mean()
    )

    resultado = receta.copy()
    resultado["producto_normalizado"] = resultado["producto_sipsa"].apply(normalizar)

    resultado = resultado.merge(
        precios,
        on="producto_normalizado",
        how="left"
    )

    resultado["costo"] = resultado.apply(
        lambda fila: calcular_costo_ingrediente(
            fila["cantidad_kg"], fila["precio_kg"]
        ) if pd.notna(fila["precio_kg"]) else float("nan"),
        axis=1
    )

    return resultado

resultado = construir_pipeline(datos_filtrados, receta)
display(resultado[["ingrediente", "producto_sipsa", "cantidad_kg", "precio_kg", "costo"]])


,ingrediente,producto_sipsa,cantidad_kg,precio_kg,costo
0,espinazo,"Carne de cerdo, espinazo",1.5,13500.0,20250.0
1,yuca,Yuca ICA,1.0,1377.0,1377.0
2,papa,Papa capira,1.0,2133.0,2133.0
3,mazorca,Chócolo mazorca,0.9,1273.0,1145.7
4,platano,Plátano dominico hartón verde,1.4,2663.0,3728.2



#10. Cálculo del costo total
***
Si todos los precios están disponibles, calculamos el costo total del sancocho para seis personas

In [ ]:

if resultado["precio_kg"].notna().all():
    costo_total = calcular_costo_total(resultado)
    print(f"Costo estimado del sancocho: ${costo_total:,.0f} COP")
else:
    sin_precio = resultado.loc[resultado["precio_kg"].isna(), "ingrediente"].tolist()
    print("No se puede calcular un total definitivo. Faltan precios para:", sin_precio)

Costo estimado del sancocho: $28,634 COP



#11. Comparación del espinazo con los demás ingredientes
***
Comparamos el costo del espinazo con el costo conjunto de la yuca, la papa, la mazorca y el plátano.

In [ ]:

if resultado["precio_kg"].notna().all():
    costo_espinazo = resultado.loc[
        resultado["ingrediente"] == "espinazo", "costo"
    ].sum()

    costo_demas = resultado.loc[
        resultado["ingrediente"] != "espinazo", "costo"
    ].sum()

    print(f"Costo del espinazo: ${costo_espinazo:,.0f} COP")
    print(f"Costo de los demás ingredientes: ${costo_demas:,.0f} COP")

    if costo_espinazo > costo_demas:
        print("El espinazo cuesta más que los demás ingredientes juntos.")
    elif costo_espinazo < costo_demas:
        print("Los demás ingredientes cuestan más que el espinazo.")
    else:
        print("Ambos costos son iguales.")

Costo del espinazo: $20,250 COP
Costo de los demás ingredientes: $8,384 COP
El espinazo cuesta más que los demás ingredientes juntos.



#12. Comparación con la predicción inicial
***
La predicción inicial del equipo fue de $45.000 COP. Comparamos esa predicción con el resultado calculado a partir de los precios SIPSA.

In [ ]:

prediccion_inicial = 45000

if resultado["precio_kg"].notna().all():
    diferencia = abs(costo_total - prediccion_inicial)
    if costo_total > prediccion_inicial:
        comparacion = "mayor"
    elif costo_total < prediccion_inicial:
        comparacion = "menor"
    else:
        comparacion = "igual"

    print(f"Predicción inicial: ${prediccion_inicial:,.0f} COP")
    print(f"Costo estimado: ${costo_total:,.0f} COP")
    print(f"El costo estimado es {comparacion} que la predicción.")
    print(f"Diferencia absoluta: ${diferencia:,.0f} COP")


Predicción inicial: $45,000 COP
Costo estimado: $28,634 COP
El costo estimado es menor que la predicción.
Diferencia absoluta: $16,366 COP



#13. Conclusiones de la sesión
***
Con los datos del archivo SIPSA mensual de 2024, se estimó el costo del sancocho usando los precios de diciembre para el mercado Manizales, Centro Galerías. El resultado depende de la correspondencia entre la receta y los productos seleccionados en SIPSA; para la papa se usó papa capira y para el plátano, dominico hartón verde.

Resultado calculado en este archivo: $28.634 COP aproximadamente para seis personas.

Comparación con la predicción de $45.000 COP: el costo calculado es menor por aproximadamente $16.366 COP.

Ingrediente o grupo más costoso: el espinazo cuesta $20.250 COP en la cantidad de 1,5 kg. Los demás ingredientes suman aproximadamente $8.384 COP, por lo que el espinazo cuesta más que los otros cuatro ingredientes juntos.

Limitación: esta base contiene solamente 2024. Para determinar si el precio de la papa o el plátano subió más en los últimos diez años, se necesitan también los datos históricos de años anteriores y comparar períodos equivalentes.